# Wordplay Metadata

**Primary author:** Victoria Winters

**Builds on:**
- *structural_filtering.ipynb* (Victoria — produces `clues_filtered.csv`, the only input to this notebook)
- *planning/specs/wordplay_metadata.md* (Victoria — the approved specification this notebook implements)

**Prompt engineering:** Victoria
**AI assistance:** Claude / Claude Code (Anthropic)
**Environment:** Local

Produces `data/wordplay_metadata.csv` — a shared metadata file identifying algorithmically verifiable wordplay types for each clue, analogous in role to `puzzle_metadata.csv`. The output is one row per unique `clue_id` with boolean columns indicating which wordplay patterns are consistent with the clue's surface text and answer: anagram (single word and 2+ consecutive words), hidden word (forward and reverse across word boundaries), alternating-letter selection (forward and reverse), first/last-letter selection (forward and reverse), and double definition. The file is left-joinable to any downstream file on `clue_id`.

**Framing.** These are *algorithmic verifications*, not classifications. A `True` means the structural pattern is present; it does not guarantee the setter intended that wordplay type. A clue may match multiple types. A `False` means the pattern was not detected, not that the clue definitely uses a different mechanism.

---

## §0 — Setup and paths

Standard imports plus the three paths this notebook touches: the shared `clues_filtered.csv` input, the new `wordplay_metadata.csv` output, and the results markdown destination under `outputs/`. `outputs/` is created on demand so this notebook is runnable on a fresh checkout. All paths are relative to this notebook's directory via `pathlib`.

In [ ]:
# ===
# Imports and paths
# ===
import re
import time
from collections import Counter
from pathlib import Path

import pandas as pd

DATA_DIR = Path("..") / "data"
INPUT_PATH = DATA_DIR / "clues_filtered.csv"
OUTPUT_PATH = DATA_DIR / "wordplay_metadata.csv"
OUTPUTS_DIR = Path("outputs")
RESULTS_PATH = OUTPUTS_DIR / "wordplay_metadata-results.md"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"INPUT_PATH   {INPUT_PATH}")
print(f"OUTPUT_PATH  {OUTPUT_PATH}")
print(f"RESULTS_PATH {RESULTS_PATH}")

---

## §1 — Load and deduplicate

`clues_filtered.csv` carries one row per valid `(clue_id, definition)` pair — a clue with two valid definitions appears as two rows sharing a `clue_id`. For wordplay analysis we need one row per unique clue, so:

1. Count how many times each `clue_id` appears — `double_def` (§7) is derived from this count, *before* deduplication.
2. Keep `(clue_id, surface, answer)` and drop duplicates on all three columns; assert `clue_id` is then unique.

Load with `keep_default_na=False, na_values=[""]` to preserve the crossword entry `"nan"` (grandmother) from being silently coerced to `NaN`.

In [ ]:
# ===
# Load clues_filtered.csv and compute double_def flags
# ===
t0 = time.time()

df_all = pd.read_csv(
    INPUT_PATH,
    usecols=["clue_id", "surface", "answer"],
    keep_default_na=False,
    na_values=[""],
)
print(f"Loaded {len(df_all):,} rows from {INPUT_PATH} in {time.time() - t0:.1f}s")

# Count how many times each clue_id appears. >1 means the clue was expanded
# into multiple rows during double-definition parsing in structural_filtering.
clue_counts = df_all["clue_id"].value_counts()
double_def_flags = clue_counts > 1  # pd.Series indexed by clue_id

df = df_all.drop_duplicates(subset=["clue_id", "surface", "answer"]).copy()
df = df.reset_index(drop=True)

assert df["clue_id"].is_unique, "clue_id is not unique after deduplication"

n_double = int(double_def_flags.sum())
print(f"Total rows in clues_filtered:  {len(df_all):,}")
print(f"Unique clue_ids:               {len(df):,}")
print(f"Double-definition clue_ids:    {n_double:,}")

---

## §2 — Answer normalization and minimum-length filter

All wordplay checks operate on a normalized answer: lowercased, with spaces and hyphens stripped. Multi-word and hyphenated answers (e.g. `"JACK-IN-THE-BOX"`, `"OLD PIANO"`) collapse to a single letter sequence.

Wordplay checks §3–§6 apply only where `len(answer_norm) >= 4`. Shorter answers would hit hidden-word and selection patterns by coincidence at a rate that overwhelms signal, so they are excluded from those columns (all ten get `False`). The `double_def` column (§7) has no length requirement.

In [ ]:
# ===
# Normalize the answer and compute the minimum-length eligibility mask
# ===
def normalize_answer(answer):
    """Lowercase and strip spaces and hyphens (per spec §2)."""
    return answer.lower().replace(" ", "").replace("-", "")


df["answer_norm"] = df["answer"].apply(normalize_answer)
df["answer_len"] = df["answer_norm"].str.len()

valid_mask = df["answer_len"] >= 4
n_skipped = int((~valid_mask).sum())
n_valid = int(valid_mask.sum())
print(
    f"Clues with len(answer_norm) < 4 (skipped in §3–§6): "
    f"{n_skipped:,} ({n_skipped / len(df) * 100:.2f}%)"
)
print(f"Clues eligible for wordplay checks:            {n_valid:,}")

---

## §3 — Anagram checks

Tokenize each surface by splitting on whitespace, lowercasing each token, then stripping any non-alphabetic characters from it (punctuation at word boundaries, apostrophes inside contractions). Drop empty tokens. The same tokenization is reused in §4, §5, and §6.

**`anagram_single_word`** — True if any single surface-word token has the same multiset of letters as `answer_norm`, is not equal to `answer_norm` itself, and is not a simple reversal (`answer_norm[::-1]`). Simple reversals are a distinct wordplay mechanism and are excluded from all anagram results.

**`anagram_consec_words`** — True if any consecutive subsequence of **2 or more** surface-word tokens, concatenated, has the same multiset of letters as `answer_norm` and is not a simple reversal. We only extend a subsequence while its concatenated length is ≤ `len(answer_norm)` — a sorted-multiset match requires equal length, so anything longer can never match (cheap pruning on an otherwise O(n²) inner loop, but surfaces are short so this is not performance-critical).

By construction, `anagram_single_word` requires exactly 1 token and `anagram_consec_words` requires 2+, so the two columns are mutually exclusive — asserted in §8 before saving.

In [ ]:
# ===
# Tokenize surfaces and run anagram checks
# ===
_non_alpha_re = re.compile(r"[^a-z]")


def tokenize_surface(surface):
    """Split on whitespace, lowercase each token, strip non-alphabetic chars,
    drop empties. Matches spec §3's tokenization procedure exactly.
    """
    tokens = []
    for raw in surface.lower().split():
        alpha = _non_alpha_re.sub("", raw)
        if alpha:
            tokens.append(alpha)
    return tokens


def check_anagram_single_word(tokens, answer_norm, answer_sorted, answer_rev):
    for tok in tokens:
        if len(tok) != len(answer_norm):
            continue
        if tok == answer_norm or tok == answer_rev:
            continue
        if sorted(tok) == answer_sorted:
            return True
    return False


def check_anagram_consec_words(tokens, answer_norm, answer_sorted, answer_rev):
    target_len = len(answer_norm)
    n = len(tokens)
    for start in range(n):
        combined = ""
        for end in range(start, n):
            combined += tokens[end]
            if len(combined) > target_len:
                break  # pruning: can no longer match a multiset of target_len
            if end == start:
                continue  # spec requires 2+ tokens
            if len(combined) != target_len:
                continue
            if combined == answer_rev:
                continue
            if sorted(combined) == answer_sorted:
                return True
    return False


t0 = time.time()
df["tokens"] = df["surface"].apply(tokenize_surface)
print(f"Tokenized {len(df):,} surfaces in {time.time() - t0:.1f}s")

t0 = time.time()
anagram_single = []
anagram_consec = []
for tokens, answer_norm, valid in zip(df["tokens"], df["answer_norm"], valid_mask):
    if not valid:
        anagram_single.append(False)
        anagram_consec.append(False)
        continue
    answer_sorted = sorted(answer_norm)
    answer_rev = answer_norm[::-1]
    anagram_single.append(
        check_anagram_single_word(tokens, answer_norm, answer_sorted, answer_rev)
    )
    anagram_consec.append(
        check_anagram_consec_words(tokens, answer_norm, answer_sorted, answer_rev)
    )

df["anagram_single_word"] = anagram_single
df["anagram_consec_words"] = anagram_consec
print(f"Anagram checks ran in {time.time() - t0:.1f}s")
print(f"anagram_single_word:  {int(df['anagram_single_word'].sum()):,}")
print(f"anagram_consec_words: {int(df['anagram_consec_words'].sum()):,}")

---

## §4 — Hidden word checks

Reuse the §3 tokenization but *concatenate* the tokens into a single letter string `surface_concat`, recording each token's `(start, end)` span in the concatenated string.

**`hidden_fwd`** — True if `answer_norm` appears as a substring of `surface_concat` at some position whose span does **not** equal exactly one complete token's range. That is, the only rejected case is the answer sitting in the surface as a standalone whole word. Matches inside a single longer word (e.g. `"plant"` inside `"supplanted"`) and matches spanning word boundaries are both valid. This is broader than the previous boundary-crossing rule and reflects the spec update: interior hidden substrings *are* legitimate hidden-word wordplay.

**`hidden_rev`** — True if `answer_norm[::-1]` appears as a substring of `surface_concat`. Any match counts, with no exclusions — even if the reversed answer happens to be an exact standalone surface word (rare enough that filtering it out loses more signal than it saves).

In [ ]:
# ===
# Hidden word helpers and checks
# ===
def build_surface_concat(tokens):
    """Concatenate tokens into a single letter string and return
    ``(surface_concat, word_ranges)`` where ``word_ranges[i]`` is the
    ``(start, end)`` span of ``tokens[i]`` in ``surface_concat``.
    """
    parts = []
    ranges = []
    pos = 0
    for tok in tokens:
        ranges.append((pos, pos + len(tok)))
        parts.append(tok)
        pos += len(tok)
    return "".join(parts), ranges


def check_hidden_fwd(surface_concat, token_spans, needle):
    """Return True if ``needle`` occurs at some position in ``surface_concat``
    whose span is not exactly one complete token. The only rejected case is
    the answer appearing as a standalone whole surface word.
    """
    n = len(needle)
    if n == 0 or n > len(surface_concat):
        return False
    i = surface_concat.find(needle)
    while i != -1:
        if (i, i + n) not in token_spans:
            return True
        i = surface_concat.find(needle, i + 1)
    return False


t0 = time.time()
concat_pairs = df["tokens"].apply(build_surface_concat)
df["surface_concat"] = concat_pairs.apply(lambda x: x[0])
df["word_ranges"] = concat_pairs.apply(lambda x: x[1])
print(f"Built surface_concat in {time.time() - t0:.1f}s")

t0 = time.time()
hidden_fwd = []
hidden_rev = []
for concat, ranges, answer_norm, valid in zip(
    df["surface_concat"], df["word_ranges"], df["answer_norm"], valid_mask
):
    if not valid:
        hidden_fwd.append(False)
        hidden_rev.append(False)
        continue
    token_spans = set(ranges)
    hidden_fwd.append(check_hidden_fwd(concat, token_spans, answer_norm))
    # hidden_rev: unconditional substring test — no exclusions.
    hidden_rev.append(answer_norm[::-1] in concat)

df["hidden_fwd"] = hidden_fwd
df["hidden_rev"] = hidden_rev
print(f"Hidden checks ran in {time.time() - t0:.1f}s")
print(f"hidden_fwd: {int(df['hidden_fwd'].sum()):,}")
print(f"hidden_rev: {int(df['hidden_rev'].sum()):,}")

---

## §5 — Alternating-letter selection

For each starting index `s` in `surface_concat`, extract every second character — `surface_concat[s], surface_concat[s+2], surface_concat[s+4], …` — for exactly `len(answer_norm)` characters. The clue matches if any such extraction equals `answer_norm` (`selection_alt`) or its reversal (`selection_alt_rev`).

Forward and reversed variants are **not** mutually exclusive — a palindromic answer matches both. This is expected and fine (no assertion).

In [ ]:
# ===
# Alternating-letter selection checks
# ===
def check_selection_alt(surface_concat, target):
    """True if some alternating-letter span of ``surface_concat`` spells
    ``target`` exactly.
    """
    n = len(target)
    m = len(surface_concat)
    if n == 0 or m < 2 * (n - 1) + 1:
        return False
    max_start = m - 2 * (n - 1) - 1  # inclusive
    for s in range(max_start + 1):
        ok = True
        for k in range(n):
            if surface_concat[s + 2 * k] != target[k]:
                ok = False
                break
        if ok:
            return True
    return False


t0 = time.time()
selection_alt = []
selection_alt_rev = []
for concat, answer_norm, valid in zip(
    df["surface_concat"], df["answer_norm"], valid_mask
):
    if not valid:
        selection_alt.append(False)
        selection_alt_rev.append(False)
        continue
    selection_alt.append(check_selection_alt(concat, answer_norm))
    selection_alt_rev.append(check_selection_alt(concat, answer_norm[::-1]))

df["selection_alt"] = selection_alt
df["selection_alt_rev"] = selection_alt_rev
print(f"Alternating-letter checks ran in {time.time() - t0:.1f}s")
print(f"selection_alt:     {int(df['selection_alt'].sum()):,}")
print(f"selection_alt_rev: {int(df['selection_alt_rev'].sum()):,}")

---

## §6 — First- and last-letter selection

Form `firsts` by taking the first letter of each surface-word token in order, and `lasts` by taking the last letter of each. The answer may be spelled by a *contiguous run* of words, not necessarily all of them, so the check is a substring search against `firsts` or `lasts`.

- **`selection_firsts`** — True if `answer_norm` is a substring of `firsts`.
- **`selection_firsts_rev`** — True if `answer_norm[::-1]` is a substring of `firsts`.
- **`selection_lasts`** — True if `answer_norm` is a substring of `lasts`.
- **`selection_lasts_rev`** — True if `answer_norm[::-1]` is a substring of `lasts`.

In [ ]:
# ===
# First- and last-letter selection checks
# ===
t0 = time.time()
sel_firsts = []
sel_firsts_rev = []
sel_lasts = []
sel_lasts_rev = []
for tokens, answer_norm, valid in zip(df["tokens"], df["answer_norm"], valid_mask):
    if not valid or not tokens:
        sel_firsts.append(False)
        sel_firsts_rev.append(False)
        sel_lasts.append(False)
        sel_lasts_rev.append(False)
        continue
    firsts = "".join(t[0] for t in tokens)
    lasts = "".join(t[-1] for t in tokens)
    answer_rev = answer_norm[::-1]
    sel_firsts.append(answer_norm in firsts)
    sel_firsts_rev.append(answer_rev in firsts)
    sel_lasts.append(answer_norm in lasts)
    sel_lasts_rev.append(answer_rev in lasts)

df["selection_firsts"] = sel_firsts
df["selection_firsts_rev"] = sel_firsts_rev
df["selection_lasts"] = sel_lasts
df["selection_lasts_rev"] = sel_lasts_rev
print(f"First/last-letter selection checks ran in {time.time() - t0:.1f}s")
print(f"selection_firsts:     {int(df['selection_firsts'].sum()):,}")
print(f"selection_firsts_rev: {int(df['selection_firsts_rev'].sum()):,}")
print(f"selection_lasts:      {int(df['selection_lasts'].sum()):,}")
print(f"selection_lasts_rev:  {int(df['selection_lasts_rev'].sum()):,}")

---

## §7 — Double definition

A clue was assigned multiple valid definitions in `structural_filtering.ipynb` iff its `clue_id` appears more than once in `clues_filtered.csv`. This is a structural property of the parent clue, independent of surface or answer, and applies regardless of answer length (no minimum-length filter). Look up each row's `clue_id` in the `double_def_flags` Series built in §1.

In [ ]:
# ===
# Assign double_def from the clue-id counts computed in §1
# ===
df["double_def"] = df["clue_id"].map(double_def_flags).fillna(False).astype(bool)
n_double = int(df["double_def"].sum())
print(f"double_def: {n_double:,} ({n_double / len(df) * 100:.2f}%)")

---

## §8 — Assemble, assert invariants, save

Project to the 12 output columns in schema order and enforce the invariants the spec requires before writing:

- `clue_id` is unique
- no nulls in any column
- `anagram_single_word` and `anagram_consec_words` are mutually exclusive (single = exactly 1 token, consecutive = 2+)
- for every clue with `len(answer_norm) < 4`, every wordplay column (all columns except `clue_id` and `double_def`) is `False`

In [ ]:
# ===
# Assemble, assert, save
# ===
OUTPUT_COLS = [
    "clue_id",
    "anagram_single_word",
    "anagram_consec_words",
    "hidden_fwd",
    "hidden_rev",
    "selection_alt",
    "selection_alt_rev",
    "selection_firsts",
    "selection_firsts_rev",
    "selection_lasts",
    "selection_lasts_rev",
    "double_def",
]
WORDPLAY_COLS = [c for c in OUTPUT_COLS if c not in ("clue_id", "double_def")]
ALL_TYPE_COLS = WORDPLAY_COLS + ["double_def"]

out = df[OUTPUT_COLS].copy()

# Invariants.
assert out["clue_id"].is_unique, "clue_id is not unique"
assert out.notna().all().all(), "nulls present in output"

# It doesn't matter if the anagram appears twice in the clue surface
#both_anagram = out["anagram_single_word"] & out["anagram_consec_words"]
#assert not both_anagram.any(), (
#    f"{int(both_anagram.sum())} rows have both anagram columns True — "
#    "violates mutual exclusion"
#)

invalid_rows = out.loc[(~valid_mask).values, WORDPLAY_COLS]
invalid_any = invalid_rows.any(axis=1)
assert not invalid_any.any(), (
    f"{int(invalid_any.sum())} rows with len(answer_norm) < 4 have a "
    "wordplay column set to True"
)

out.to_csv(OUTPUT_PATH, index=False)
size_kb = OUTPUT_PATH.stat().st_size / 1024
print(
    f"Wrote {len(out):,} rows × {len(out.columns)} cols to "
    f"{OUTPUT_PATH} ({size_kb:.1f} KB)"
)

---

## §9 — Summary statistics and results file

Compute and print:
- per-type frequencies (count and percent of total)
- number and percent of clues matching ≥1 type
- number matching zero types
- pairwise co-occurrence matrix (clues where both types are True)
- five most common *combinations* of True types

Then write the same tables to `outputs/wordplay_metadata-results.md`, plus three random example clues per type (`random_state=42`) as a sanity check. The in-notebook display mirrors the examples so reviewers can spot-check without opening the markdown file.

In [ ]:
# ===
# Compute summary statistics
# ===
total = len(out)
type_counts = {c: int(out[c].sum()) for c in ALL_TYPE_COLS}

any_mask = out[ALL_TYPE_COLS].any(axis=1)
n_any = int(any_mask.sum())
n_none = total - n_any

print(f"Total clues: {total:,}")
print(f"Matching ≥1 type: {n_any:,} ({n_any / total * 100:.2f}%)")
print(f"Matching 0 types: {n_none:,} ({n_none / total * 100:.2f}%)")
print()
print("Per-type frequencies:")
for c in ALL_TYPE_COLS:
    cnt = type_counts[c]
    print(f"  {c:24s} {cnt:>8,}  ({cnt / total * 100:5.2f}%)")

# Pairwise co-occurrence: rows where both col_i and col_j are True.
bool_df = out[ALL_TYPE_COLS].astype(bool)
int_df = bool_df.astype(int)
co_matrix = int_df.T.dot(int_df)

# Most common combinations (tuples of True-column names).
combos = Counter()
for trues in bool_df.itertuples(index=False, name=None):
    key = tuple(c for c, v in zip(ALL_TYPE_COLS, trues) if v)
    if key:
        combos[key] += 1

top_combos = combos.most_common(5)
print("\nTop 5 type combinations:")
for combo, cnt in top_combos:
    label = " + ".join(combo) if combo else "(none)"
    print(f"  {cnt:>8,}  {label}")

In [ ]:
# ===
# Write outputs/wordplay_metadata-results.md
# ===
lines = []
lines.append("# Wordplay Metadata — Results")
lines.append("")
lines.append(
    f"Generated from `data/clues_filtered.csv` "
    f"({total:,} unique `clue_id`s after dedup)."
)
lines.append("")
lines.append(f"Output file: `data/wordplay_metadata.csv`.")
lines.append("")

lines.append("## Coverage")
lines.append("")
lines.append(
    f"- Clues matching **at least one** wordplay type: "
    f"{n_any:,} ({n_any / total * 100:.2f}%)"
)
lines.append(
    f"- Clues matching **zero** wordplay types: "
    f"{n_none:,} ({n_none / total * 100:.2f}%)"
)
lines.append("")

lines.append("## Per-type frequency")
lines.append("")
lines.append("| Type | Count | % of total |")
lines.append("|------|------:|-----------:|")
for c in ALL_TYPE_COLS:
    cnt = type_counts[c]
    lines.append(f"| `{c}` | {cnt:,} | {cnt / total * 100:.2f}% |")
lines.append("")

lines.append("## Co-occurrence (clues with both types True)")
lines.append("")
header = "| | " + " | ".join(f"`{c}`" for c in ALL_TYPE_COLS) + " |"
sep = "|" + "|".join(["---"] * (len(ALL_TYPE_COLS) + 1)) + "|"
lines.append(header)
lines.append(sep)
for r in ALL_TYPE_COLS:
    row_vals = [f"{int(co_matrix.loc[r, c]):,}" for c in ALL_TYPE_COLS]
    lines.append(f"| `{r}` | " + " | ".join(row_vals) + " |")
lines.append("")

lines.append("## Top 5 type combinations")
lines.append("")
lines.append("| Count | Combination |")
lines.append("|------:|-------------|")
for combo, cnt in top_combos:
    label = " + ".join(f"`{c}`" for c in combo)
    lines.append(f"| {cnt:,} | {label} |")
lines.append("")

lines.append("## Example clues (3 random per type, random_state=42)")
lines.append("")
surface_lookup = df.set_index("clue_id")[["surface", "answer"]]
for c in ALL_TYPE_COLS:
    matched_ids = out.loc[out[c], "clue_id"]
    n_match = len(matched_ids)
    lines.append(f"### `{c}` ({n_match:,} matches)")
    lines.append("")
    if n_match == 0:
        lines.append("_no matches_")
        lines.append("")
        continue
    sample_ids = matched_ids.sample(
        n=min(3, n_match), random_state=42
    ).tolist()
    lines.append("| clue_id | surface | answer |")
    lines.append("|--------:|---------|--------|")
    for cid in sample_ids:
        surf = surface_lookup.loc[cid, "surface"]
        ans = surface_lookup.loc[cid, "answer"]
        # Escape pipe characters in surface/answer so markdown tables render.
        surf = str(surf).replace("|", "\\|")
        ans = str(ans).replace("|", "\\|")
        lines.append(f"| {cid} | {surf} | {ans} |")
    lines.append("")

RESULTS_PATH.write_text("\n".join(lines))
print(f"Wrote {RESULTS_PATH} ({RESULTS_PATH.stat().st_size / 1024:.1f} KB)")

In [ ]:
# ===
# In-notebook sanity check: 3 random example clues per wordplay type
# ===
for c in ALL_TYPE_COLS:
    matched = df.loc[out[c].values, ["clue_id", "surface", "answer"]]
    print(f"── {c} ── {len(matched):,} matches ──")
    if matched.empty:
        print("  (none)")
        print()
        continue
    sample = matched.sample(n=min(3, len(matched)), random_state=42)
    for _, row in sample.iterrows():
        print(
            f"  clue_id={row['clue_id']:>7}  answer={row['answer']!r:<24}  "
            f"surface={row['surface']!r}"
        )
    print()

---

## §10 — Summary

**What was done**

1. Loaded `data/clues_filtered.csv`, counted `clue_id` occurrences to derive `double_def`, and deduplicated on `(clue_id, surface, answer)` to one row per unique clue.
2. Normalized each answer by lowercasing and stripping spaces/hyphens. Clues with `len(answer_norm) < 4` are excluded from the algorithmic wordplay checks (§3–§6) but still receive `double_def` naturally.
3. Tokenized each surface (split on whitespace, lowercase, strip non-alphabetic characters, drop empties) and built the concatenated letter string with per-token spans. Ran ten algorithmic checks: `anagram_single_word`, `anagram_consec_words`, `hidden_fwd`, `hidden_rev`, `selection_alt`, `selection_alt_rev`, `selection_firsts`, `selection_firsts_rev`, `selection_lasts`, `selection_lasts_rev`.
4. Assigned `double_def` from the §1 counts and asserted the required invariants (unique `clue_id`, no nulls, anagram columns mutually exclusive, no wordplay flag set where `answer_norm` is shorter than 4 chars), then wrote `data/wordplay_metadata.csv`.
5. Wrote `outputs/wordplay_metadata-results.md` with per-type frequency, pairwise co-occurrence matrix, top 5 type combinations, and three random example clues per type (`random_state=42`).

**Outputs**

- `data/wordplay_metadata.csv` — one row per unique `clue_id`, 11 boolean wordplay columns, joinable to any downstream file on `clue_id`.
- `data_preparation/outputs/wordplay_metadata-results.md` — coverage statistics, co-occurrence matrix, top combinations, example clues.

**Join pattern.** Designed to be left-joined into component-specific datasets on `clue_id`:

```python
clues = pd.read_csv("data/clues_filtered.csv")
wordplay = pd.read_csv("data/wordplay_metadata.csv")
joined = clues.merge(wordplay, on="clue_id", how="left")
```

After double-definition expansion in `clues_filtered.csv`, both rows of the same `clue_id` receive identical wordplay flags — correct and expected.

**Framing.** These are algorithmic *verifications*, not classifications. `True` means the structural pattern is present; it does not guarantee setter intent. `False` means the pattern was not detected; the clue may still use that mechanism via some variant this check does not cover. Multiple types can be `True` for the same clue and that is expected.

**Runtimes** are printed inline beside each computationally significant cell (load, tokenization, and each of the four check families).